# Desafio Fine-tuning — Classificador de mensagens (venda × suporte)

Fine-tuning do **BERTimbau** (`neuralmind/bert-base-portuguese-cased`), um BERT treinado em português, para classificar mensagens do bot de atendimento em `venda` ou `suporte`.

**Como usar no Colab:** ative a GPU (*Ambiente de execução → Alterar tipo → T4 GPU*), rode as células em ordem e, quando pedido, faça upload de `treino.jsonl` e `teste.jsonl`.

## 1. Instalação

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

## 2. Upload e leitura dos dados

O leitor abaixo aceita JSONL normal (um objeto por linha) e também objetos colados na mesma linha, por segurança.

In [ ]:
import json, os
from google.colab import files

if not (os.path.exists("treino.jsonl") and os.path.exists("teste.jsonl")):
    print("Selecione treino.jsonl e teste.jsonl:")
    files.upload()

def read_jsonl(path):
    text = open(path, encoding="utf-8").read()
    decoder, idx, rows = json.JSONDecoder(), 0, []
    while idx < len(text):
        while idx < len(text) and text[idx].isspace():
            idx += 1
        if idx >= len(text):
            break
        obj, idx = decoder.raw_decode(text, idx)
        rows.append(obj)
    return rows

train_rows = read_jsonl("treino.jsonl")
test_rows = read_jsonl("teste.jsonl")
print(len(train_rows), "exemplos de treino |", len(test_rows), "de teste")
print(train_rows[:2])

## 3. Exploração rápida

In [ ]:
import pandas as pd

df_train = pd.DataFrame(train_rows).rename(columns={"prompt": "text", "completion": "label"})
df_test = pd.DataFrame(test_rows).rename(columns={"prompt": "text", "completion": "label"})

print("Distribuição (treino):\n", df_train["label"].value_counts(), "\n")
print("Distribuição (teste):\n", df_test["label"].value_counts(), "\n")
print("Duplicados no treino:", df_train.duplicated("text").sum())
print("Textos do teste que também estão no treino (vazamento):", df_test["text"].isin(df_train["text"]).sum())
print("Tamanho médio (palavras):", df_train["text"].str.split().str.len().mean().round(1))

## 4. Labels e datasets

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

labels = sorted(df_train["label"].unique())          # ['suporte', 'venda']
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
print(label2id)

for d in (df_train, df_test):
    d["label"] = d["label"].map(label2id)

# o desafio dá treino e teste; criamos uma validação a partir do treino
tr, val = train_test_split(df_train, test_size=0.1, random_state=42, stratify=df_train["label"])

ds_train = Dataset.from_pandas(tr.reset_index(drop=True))
ds_val = Dataset.from_pandas(val.reset_index(drop=True))
ds_test = Dataset.from_pandas(df_test.reset_index(drop=True))

## 5. Tokenização

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

ds_train = ds_train.map(tokenize, batched=True)
ds_val = ds_val.map(tokenize, batched=True)
ds_test = ds_test.map(tokenize, batched=True)

## 6. Modelo, métricas e treinamento

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id
)

def compute_metrics(eval_pred):
    logits, y = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(y, preds, average="weighted", zero_division=0)
    return {"accuracy": accuracy_score(y, preds), "precision": p, "recall": r, "f1": f1}

args = TrainingArguments(
    output_dir="bertimbau-venda-suporte",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=20,
    report_to="none",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 7. Avaliação no `teste.jsonl`

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred = trainer.predict(ds_test)
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids

print(classification_report(y_true, y_pred, target_names=labels, digits=4))

cm = confusion_matrix(y_true, y_pred)
print(pd.DataFrame(cm, index=[f"real:{l}" for l in labels], columns=[f"prev:{l}" for l in labels]))

## 8. Erros do modelo (pra análise)

In [ ]:
errors = df_test.assign(pred=y_pred)
errors = errors[errors["label"] != errors["pred"]].copy()
errors["label"] = errors["label"].map(id2label)
errors["pred"] = errors["pred"].map(id2label)
print(f"{len(errors)} erros de {len(df_test)}")
errors.head(15)

## 9. Testando com mensagens novas

In [ ]:
from transformers import pipeline

clf = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer,
               device=0 if trainer.model.device.type == "cuda" else -1)

exemplos = [
    "Quero comprar a geladeira inox de 400 litros, tem em estoque?",
    "Minha máquina de lavar parou de centrifugar, o que pode ser?",
    "Qual o preço do aspirador robô?",
    "Como faço para resetar a TV Smart?",
    "Vocês parcelam o sofá em 10x?",
]
for t, r in zip(exemplos, clf(exemplos)):
    print(f"{r['label']:8} ({r['score']:.2%})  <-  {t}")

## 10. Salvando o modelo

In [ ]:
trainer.save_model("bertimbau-venda-suporte-final")
tokenizer.save_pretrained("bertimbau-venda-suporte-final")

# opcional: baixar zipado
!zip -qr bertimbau-venda-suporte-final.zip bertimbau-venda-suporte-final
# files.download("bertimbau-venda-suporte-final.zip")

## Análise final

Preencha após rodar:

- **Resultado:** acurácia / F1 no teste.
- **Erros:** que tipo de mensagem confundiu o modelo? (ex.: "queria saber como funciona" pode ser venda ou suporte)
- **Limitação:** o dataset é sintético, então métricas muito altas podem não se repetir com mensagens reais.
- **Próximos passos:** mais dados reais, mais classes (troca, cancelamento), comparar com outro modelo (ex.: DistilBERT multilíngue).